In [ ]:
from uavcalibration.datasets import *
from uavcalibration.map import *
import uavcalibration.calibrate as calibrate
from uavcalibration.pipeline import *
from uavcalibration.calibrate_pipeline import *

In [ ]:
test_size = 50

maxsize = 10
tolerance = 5.0

dataset = VisLocDataset(r"..\datasets\UAV_VisLoc_example")
satellite_map = GeoTiffMap([i.image_path for i in dataset.satellite_infos.values()])

calibrator = calibrate.Calibrator(satellite_map, tolerance, plot=False)
async with satellite_map:
    # warm up
    uav_data = dataset[0]
    print(await calibrator.calibrate(uav_data))

In [ ]:
from uavcalibration.calibrate import CalibrateCTX


class Consumer(SyncStage[calibrate.CalibrateCTX, None, None, None]):
    def __init__(self, input_maxsize: int = 10) -> None:
        super().__init__(input_maxsize)
        self.count = 0

    def preprocess(self, i: CalibrateCTX) -> None:
        self.count += 1
        print(self.count, i.final_transform.resolution)


pipeline = create_pipeline(satellite_map, maxsize, tolerance)
pipeline.append(Consumer(maxsize).worker())
async with satellite_map, pipeline:
    for i in range(test_size):
        uav_data = dataset[i]
        await pipeline.add_input(uav_data)

In [ ]:
calibrator = calibrate.Calibrator(satellite_map, tolerance, plot=False)
count = 0
async with satellite_map, pipeline:
    for i in range(test_size):
        uav_data = dataset[i]
        await calibrator.calibrate(uav_data)
        count += 1
        print(count)